In [1]:
import tiktoken

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokerizer_list = tiktoken.list_encoding_names()
# print(tokerizer_list)
# ['gpt2', 'r50k_base', 'p50k_base', 'p50k_edit', 'cl100k_base', 'o200k_base', 'o200k_harmony']
tokenizer = tiktoken.get_encoding("gpt2")

enc_text = tokenizer.encode(raw_text)
print(len(enc_text)) # 5145

5145


In [2]:
# 조금 더 흥미로운 텍스트 구절을 만들기 위해 데이터셋에 있는 처음 50개 토큰을 삭제

enc_sample = enc_text[50:]

In [3]:
# 다음 단어 예측 작업을 위해 입력-타깃 쌍을 만드는 가장 쉽고 직관적인 방법 중 하나는 입력 토큰을 담은 x와 입력에서 토큰 하나만큼 이동한 타깃을 담은 y 변수를 만드는 것

context_size = 4 # 문맥 크기는 입력에 얼마나 많은 토큰을 포함할지 결정
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [4]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [5]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [8]:
# 코드 2-5 배치 입력과 타깃을 위한 데이터셋

import torch
from torch.utils.data import Dataset, DataLoader

# PyTorch의 Dataset을 상속받아 "나만의 데이터셋"을 정의한다.
# Dataset을 상속하면 나중에 DataLoader가 이 클래스를 자동으로 배치 단위로
# 꺼내 쓸 수 있게 된다. 이를 위해 __len__과 __getitem__ 두 메서드만
# 규칙에 맞게 구현해주면 된다.
class GPTDatasetV1(Dataset):

    # 객체를 만들 때 딱 한 번 실행되는 초기화 함수.
    #   txt        : 원본 텍스트(문자열) 전체
    #   tokenizer  : 텍스트를 토큰 ID로 바꿔줄 토크나이저 (예: tiktoken gpt2)
    #   max_length : 입력 한 덩어리에 넣을 토큰 개수 (= 문맥 크기, context size)
    #   stride     : 다음 덩어리를 만들 때 몇 칸씩 건너뛸지 (창문을 옆으로 미는 폭)
    def __init__(self, txt, tokenizer, max_length, stride):
        # 잘라 만든 입력 덩어리들을 담아둘 빈 리스트
        self.input_ids = []
        # 각 입력에 대응하는 타깃(정답) 덩어리들을 담아둘 빈 리스트
        self.target_ids = []

        # 1) 텍스트 전체를 토큰 ID의 긴 리스트로 한 번에 변환한다.
        #    예: "Hello world ..." -> [15496, 995, ...]
        token_ids = tokenizer.encode(txt)

        # 2) 긴 토큰 리스트를 창문(window)을 옆으로 밀며 여러 덩어리로 자른다.
        #    - 시작 0에서부터 stride 간격으로 i를 이동시킨다.
        #    - 끝을 (len - max_length)로 잡는 이유:
        #      타깃은 입력보다 한 칸 뒤까지 필요하므로(i + max_length + 1),
        #      마지막에 범위를 벗어나지 않도록 여유를 둔 것.
        for i in range(0, len(token_ids) - max_length, stride):
            # 입력 덩어리: i번째부터 max_length개
            #   예) i=0, max_length=4 -> token_ids[0:4] = [t0, t1, t2, t3]
            input_chunk = token_ids[i:i + max_length]

            # 타깃 덩어리: 입력보다 딱 한 칸 뒤로 밀린 max_length개
            #   예) token_ids[1:5] = [t1, t2, t3, t4]
            #   -> 입력 t0을 보면 t1을 맞히고, t0,t1을 보면 t2를 맞히는 구조
            target_chunk = token_ids[i + 1:i + max_length + 1]

            # 파이썬 리스트를 파이토치 텐서로 바꿔서 리스트에 저장한다.
            # (모델은 텐서 형태의 숫자 묶음을 입력으로 받기 때문)
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # 이 데이터셋에 (입력, 타깃) 쌍이 총 몇 개 들어있는지 반환한다.
    # DataLoader가 "전체 개수"를 알아야 몇 번 반복할지 정할 수 있어서 필요하다.
    def __len__(self):
        return len(self.input_ids)

    # idx번째 (입력, 타깃) 쌍을 하나 꺼내 반환한다.
    # DataLoader가 배치를 만들 때 이 함수를 인덱스별로 호출해서 데이터를 모은다.
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [10]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

# 하나의 배치 (batch_size=4)
# ├─ 묶음1: 입력[256개] + 타깃[256개]   (시작 0)
# ├─ 묶음2: 입력[256개] + 타깃[256개]   (시작 128)
# ├─ 묶음3: 입력[256개] + 타깃[256개]   (시작 256)
# └─ 묶음4: 입력[256개] + 타깃[256개]   (시작 384)

In [11]:
# 문맥 크기를 4와 배치 크기 1로 dataloader를 테스트

with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [12]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [15]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("입력: \n", inputs)
print("\n타깃: \n", targets)

입력: 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

타깃: 
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
